# Phase 2 — LegalSeq2Seq NER (P1_S3_EncoderDecoder176M)

In [3]:
%pip -q install tokenizers cloudpickle

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os ,sys ,json ,math ,time ,random ,hashlib ,bisect ,socket ,subprocess ,zipfile ,gc ,shutil 
from pathlib import Path 
from collections import Counter 
from contextlib import nullcontext 
from datetime import timedelta 
import numpy as np 
import torch 
from torch import nn 
from torch .nn import functional as F 
from torch .utils .checkpoint import checkpoint 
from torch .utils .data import Dataset ,DataLoader ,DistributedSampler 
from torch .nn .parallel import DistributedDataParallel as DDP 
import torch .distributed as dist 
import cloudpickle 
from tokenizers import Tokenizer 

BASE =Path ('/kaggle/input/datasets/raaz74/p2-a2')
DATA_DIR =Path ('/kaggle/input/datasets/raaz74/p2-a2/Phase_2_LNER')
OUT =Path ('/kaggle/working/phase2_encoder_decoder_six_methods')
OUT .mkdir (parents =True ,exist_ok =True )

C =json .loads ((BASE /'config.json').read_text ())
TOK =Tokenizer .from_file (str (BASE /'tokenizer.json'))
TOK .no_padding ()
TOK .no_truncation ()
C ['pad_id']=TOK .token_to_id ('<pad>')
LABELS =(DATA_DIR /'labels.txt').read_text ().split ()
L2I ={x :i for i ,x in enumerate (LABELS )}
TAGS =['O']+[p +'-'+l for l in LABELS for p in ['B','I']]
R =dict (seq =512 ,stride =320 ,micro =1 ,accum =4 ,epochs =12 ,stop_epoch =12 ,
lr =2e-5 ,head_lr =2e-4 ,wd =.01 ,warmup =.1 ,patience =3 ,seed =781 ,
dropout =.1 ,boundary_weight =.25 ,balanced =False ,causal =False ,
prompt_tokens =16 ,method ='lorma',rank =8 ,alpha =16 ,delta =16 ,teacher =None ,kd =.5 ,temperature =2. ,
student =False ,partial_encoder =2 ,partial_decoder =3 ,base =str (BASE ),out =str (OUT ),prepared =str (OUT /'prepared.pkl'))
for filename in ['config.json','tokenizer.json']:
    shutil .copyfile (BASE /filename ,OUT /filename )
(OUT /'labels.txt').write_text ('\n'.join (LABELS ))

71

## 1. Documents and split

In [7]:
texts ,gold ,folds ={},{},{}
for f in [1 ,2 ]:
    t =json .loads ((DATA_DIR /f'fold{f }-text.json').read_text ())
    g =json .loads ((DATA_DIR /f'fold{f }-gold.json').read_text ())
    if set (t )!=set (g )or set (t )&set (texts ):
        raise ValueError ('Text/gold IDs disagree or folds reuse an ID.')
    texts .update (t );gold .update (g );folds .update ({k :f for k in t })
test =json .loads ((DATA_DIR /'fold3-text.json').read_text ())
counts =Counter (e [2 ]for rows in gold .values ()for e in rows )
for k ,t in texts .items ():
    last =-1 
    for s ,e ,l ,*extra in sorted (gold [k ]):
        v =extra [0 ]if extra else t [s :e ]
        if not (0 <=s <e <=len (t )and t [s :e ]==v and l in L2I and s >=last ):
            raise ValueError (f'Invalid/overlapping gold annotation in {k }: {(s ,e ,l )}')
        last =e 
hashes =[hashlib .sha256 (t .encode ()).hexdigest ()for t in texts .values ()]
if len (set (hashes ))!=len (hashes ):
    raise ValueError ('Group duplicate documents before splitting.')
ids =sorted (texts )
entity_counts =np .array ([[sum (e [2 ]==l for e in gold [k ])for l in LABELS ]for k in ids ])
rng =np .random .default_rng (R ['seed']);best_cost =float ('inf')
for _ in range (2000 ):
    v =sorted (sum ([rng .choice ([k for k in ids if folds [k ]==f ],7 ,replace =False ).tolist ()for f in [1 ,2 ]],[]))
    mask =np .array ([k in v for k in ids ]);vc =entity_counts [mask ].sum (0 )
    if (vc ==0 ).any ()or (entity_counts [~mask ].sum (0 )==0 ).any ():
        continue 
    cost =np .mean ((vc /entity_counts .sum (0 )-.2 )**2 )
    if cost <best_cost :
        best_cost ,valid_ids =cost ,v 
train_ids =[k for k in ids if k not in valid_ids ]
split =dict (train =train_ids ,valid =valid_ids ,seed =R ['seed'])
(OUT /'split.json').write_text (json .dumps (split ,indent =2 ))
print (f'{len (texts )} labelled documents | {sum (counts .values ()):,} entities | {len (test )} test documents')
print ('Train / validation:',len (train_ids ),len (valid_ids ))
print ('Entities by class:',dict (counts ))

70 labelled documents | 8,393 entities | 35 test documents
Train / validation: 56 14
Entities by class: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}


## 2. BIO labels and exact boundaries

In [8]:
def file_hash (path ):
    digest =hashlib .sha256 ()
    with open (path ,'rb')as f :
        for block in iter (lambda :f .read (8 *1024 *1024 ),b''):
            digest .update (block )
    return digest .hexdigest ()

def encode_doc (text ,entities ):
    enc =TOK .encode (text ,add_special_tokens =False )
    off =np .array (enc .offsets ,dtype =np .int32 ).reshape (-1 ,2 )
    starts ,ends =off [:,0 ].tolist (),off [:,1 ].tolist ()
    y =np .zeros (len (off ),dtype =np .int64 )
    ds =np .full (len (off ),-100 ,dtype =np .int64 );de =ds .copy ()
    spans =[]
    for s ,e ,l ,*_ in entities :
        a =bisect .bisect_right (ends ,s );b =bisect .bisect_left (starts ,e )-1 
        if not (0 <=a <=b <len (off )and starts [a ]<=s <ends [a ]and starts [b ]<e <=ends [b ]):
            raise ValueError ('Gold endpoint is not contained in a BPE token.')
        if y [a :b +1 ].any ():
            raise ValueError ('Two entities share a token; use a span head for this data.')
        ds [a ],de [b ]=s -starts [a ],ends [b ]-e 
        if max (ds [a ],de [b ])>=R ['delta']:
            raise ValueError ('Increase boundary head size and repeat the alignment audit.')
        y [a ]=1 +2 *L2I [l ];y [a +1 :b +1 ]=2 +2 *L2I [l ]
        spans .append ((a ,b ,L2I [l ],s ,e ))
    return dict (text =text ,ids =np .array (enc .ids ,dtype =np .int64 ),off =off ,y =y ,ds =ds ,de =de ,spans =spans )

def windows (n ,seq ,stride ):
    if n ==0 :return []
    starts =list (range (0 ,max (1 ,n -seq +1 ),stride ))
    if starts [-1 ]+seq <n :
        starts .append (n -seq )
    ends =[min (n ,s +seq )for s in starts ]
    cuts =[0 ]+[(ends [i -1 ]+starts [i ])//2 for i in range (1 ,len (starts ))]+[n ]
    return [(s ,e ,cuts [i ],cuts [i +1 ])for i ,(s ,e )in enumerate (zip (starts ,ends ))]

max_delta =0 
for key in train_ids :
    offsets =TOK .encode (texts [key ],add_special_tokens =False ).offsets 
    starts ,ends =[p [0 ]for p in offsets ],[p [1 ]for p in offsets ]
    for start ,end ,*_ in gold [key ]:
        left =bisect .bisect_right (ends ,start )
        right =bisect .bisect_left (starts ,end )-1 
        max_delta =max (max_delta ,start -starts [left ],ends [right ]-end )
R ['delta']=max (16 ,max_delta +1 )

records ={k :encode_doc (t ,gold .get (k ,[]))for k ,t in {**texts ,**test }.items ()}
if set (texts )&set (test ):
    raise ValueError ('Training and test IDs overlap.')
trim_exact =0 ;recovered ={};max_span =0 
for k in ids :
    r =records [k ];recovered [k ]=[]
    owner =np .zeros (len (r ['ids']),dtype =np .int32 )
    for s ,e ,a ,b in windows (len (owner ),R ['seq'],R ['stride']):
        owner [a :b ]+=1 
        for x ,z ,_ ,_ ,_ in r ['spans']:
            if x <b and z >=a and not (s <=x <=z <e ):
                raise ValueError ('Increase overlap: a supervised entity is cut by its context window.')
    if not np .all (owner ==1 ):raise ValueError ('Window ownership is not exact.')
    for a ,b ,l ,s ,e in r ['spans']:
        ps =int (r ['off'][a ,0 ]+r ['ds'][a ]);pe =int (r ['off'][b ,1 ]-r ['de'][b ])
        recovered [k ].append ([ps ,pe ,LABELS [l ]])
        q ,w =int (r ['off'][a ,0 ]),int (r ['off'][b ,1 ])
        while q <w and r ['text'][q ].isspace ():q +=1 
        while w >q and r ['text'][w -1 ].isspace ():w -=1 
        trim_exact +=(q ,w )==(s ,e );max_span =max (max_span ,b -a +1 )
    if {tuple (x )for x in recovered [k ]}!={tuple (x [:3 ])for x in gold [k ]}:
        raise ValueError ('Gold span round-trip failed.')
print (f'Exact round-trip: {sum (map (len ,recovered .values ())):,}/{sum (counts .values ()):,}; trimmed: {trim_exact :,}; max entity: {max_span } tokens')
manifest =dict (base_checkpoint_sha256 =file_hash (BASE /'best.pt'),
tokenizer_sha256 =hashlib .sha256 ((BASE /'tokenizer.json').read_bytes ()).hexdigest (),
data_sha256 ={p .name :hashlib .sha256 (p .read_bytes ()).hexdigest ()for p in DATA_DIR .glob ('*')if p .is_file ()},
split =split ,counts =dict (counts ),trimmed_exact =trim_exact ,max_entity_tokens =max_span )
(OUT /'manifest.json').write_text (json .dumps (manifest ,indent =2 ))
with (OUT /'prepared.pkl').open ('wb')as f :
    cloudpickle .dump (dict (records =records ,gold =gold ,split =split ,labels =LABELS ,config =C ,manifest =manifest ),f )

Exact round-trip: 8,393/8,393; trimmed: 8,241; max entity: 52 tokens


## 3. LegalSeq2Seq encoder–decoder and LoRMA+

In [9]:
import math 
import torch 
import torch .nn as nn 
import torch .nn .functional as F 
from torch .utils .checkpoint import checkpoint 

class RMSNorm (nn .Module ):
    def __init__ (self ,dim ):
        super ().__init__ ()
        self .weight =nn .Parameter (torch .ones (dim ))

    def forward (self ,x ):
        z =x .float ()
        z =z *torch .rsqrt (z .square ().mean (-1 ,keepdim =True )+1e-5 )
        return z .to (x .dtype )*self .weight 

class SelfAttention (nn .Module ):
    def __init__ (self ,cfg ,causal ):
        super ().__init__ ()
        self .heads =cfg ['heads']
        self .head_dim =cfg ['dim']//self .heads 
        self .causal =causal 
        self .qkv =nn .Linear (cfg ['dim'],3 *cfg ['dim'],bias =False )
        self .out =nn .Linear (cfg ['dim'],cfg ['dim'],bias =False )
        inv =10000 **(-torch .arange (0 ,self .head_dim ,2 ).float ()/self .head_dim )
        self .register_buffer ('inv_freq',inv ,persistent =False )

    def rope (self ,x ,positions ):
        phase =positions .float ().unsqueeze (-1 )*self .inv_freq .float ()
        cos ,sin =phase .cos ()[None ,None ].to (x .dtype ),phase .sin ()[None ,None ].to (x .dtype )
        even ,odd =x [...,::2 ],x [...,1 ::2 ]
        return torch .stack ((even *cos -odd *sin ,even *sin +odd *cos ),-1 ).flatten (-2 )

    def forward (self ,x ,mask =None ,past =None ,use_cache =False ):
        b ,t ,d =x .shape 
        q ,k ,v =self .qkv (x ).view (b ,t ,3 ,self .heads ,self .head_dim ).unbind (2 )
        q ,k ,v =(z .transpose (1 ,2 )for z in (q ,k ,v ))
        old =0 if past is None else past [0 ].size (2 )
        positions =torch .arange (old ,old +t ,device =x .device )
        q ,k =self .rope (q ,positions ),self .rope (k ,positions )
        if past is not None :
            k ,v =torch .cat ((past [0 ],k ),2 ),torch .cat ((past [1 ],v ),2 )
        allowed =None if mask is None else mask [:,None ,None ,:]
        fast_causal =self .causal and past is None and mask is None 
        if self .causal and not fast_causal :
            causal_mask =torch .arange (k .size (2 ),device =x .device )[None ,:]<=positions [:,None ]
            allowed =causal_mask [None ,None ]if allowed is None else allowed &causal_mask [None ,None ]
        y =F .scaled_dot_product_attention (q ,k ,v ,attn_mask =allowed ,
        is_causal =fast_causal ,dropout_p =0.0 )
        y =self .out (y .transpose (1 ,2 ).contiguous ().view (b ,t ,d ))
        return y ,(k ,v )if use_cache else None 

class CrossAttention (nn .Module ):
    def __init__ (self ,cfg ):
        super ().__init__ ()
        self .heads ,self .head_dim =cfg ['heads'],cfg ['dim']//cfg ['heads']
        self .q =nn .Linear (cfg ['dim'],cfg ['dim'],bias =False )
        self .kv =nn .Linear (cfg ['dim'],2 *cfg ['dim'],bias =False )
        self .out =nn .Linear (cfg ['dim'],cfg ['dim'],bias =False )

    def project_memory (self ,memory ):
        b ,s ,_ =memory .shape 
        k ,v =self .kv (memory ).view (b ,s ,2 ,self .heads ,self .head_dim ).unbind (2 )
        return k .transpose (1 ,2 ),v .transpose (1 ,2 )

    def forward (self ,x ,memory ,mask ,cached =None ):
        b ,t ,d =x .shape 
        q =self .q (x ).view (b ,t ,self .heads ,self .head_dim ).transpose (1 ,2 )
        k ,v =self .project_memory (memory )if cached is None else cached 
        y =F .scaled_dot_product_attention (q ,k ,v ,attn_mask =mask [:,None ,None ,:],dropout_p =0.0 )
        return self .out (y .transpose (1 ,2 ).contiguous ().view (b ,t ,d ))

class FeedForward (nn .Module ):
    def __init__ (self ,cfg ):
        super ().__init__ ()
        d ,f =cfg ['dim'],cfg ['ffn_dim']
        self .gate =nn .Linear (d ,f ,bias =False )
        self .up =nn .Linear (d ,f ,bias =False )
        self .down =nn .Linear (f ,d ,bias =False )

    def forward (self ,x ):
        return self .down (F .silu (self .gate (x ))*self .up (x ))

class EncoderBlock (nn .Module ):
    def __init__ (self ,cfg ):
        super ().__init__ ()
        self .norm1 ,self .norm2 =RMSNorm (cfg ['dim']),RMSNorm (cfg ['dim'])
        self .attn =SelfAttention (cfg ,causal =False )
        self .ffn =FeedForward (cfg )

    def forward (self ,x ,mask ):
        x =x +self .attn (self .norm1 (x ),mask )[0 ]
        return x +self .ffn (self .norm2 (x ))

class DecoderBlock (nn .Module ):
    def __init__ (self ,cfg ):
        super ().__init__ ()
        self .norm1 ,self .norm2 ,self .norm3 =(RMSNorm (cfg ['dim'])for _ in range (3 ))
        self .attn =SelfAttention (cfg ,causal =True )
        self .cross =CrossAttention (cfg )
        self .ffn =FeedForward (cfg )

    def forward (self ,x ,memory ,source_mask ,target_mask =None ,past =None ,cross_kv =None ,use_cache =False ):
        a ,cache =self .attn (self .norm1 (x ),target_mask ,past ,use_cache )
        x =x +a 
        x =x +self .cross (self .norm2 (x ),memory ,source_mask ,cross_kv )
        return x +self .ffn (self .norm3 (x )),cache 

class LegalSeq2Seq (nn .Module ):
    def __init__ (self ,cfg ):
        super ().__init__ ()
        self .cfg =dict (cfg )
        self .embed =nn .Embedding (cfg ['vocab_size'],cfg ['dim'])
        self .encoder =nn .ModuleList ([EncoderBlock (cfg )for _ in range (cfg ['encoder_layers'])])
        self .decoder =nn .ModuleList ([DecoderBlock (cfg )for _ in range (cfg ['decoder_layers'])])
        self .encoder_norm ,self .decoder_norm =RMSNorm (cfg ['dim']),RMSNorm (cfg ['dim'])
        self .apply (self ._init )
        for block in self .encoder :
            for weight in (block .attn .out .weight ,block .ffn .down .weight ):
                nn .init .normal_ (weight ,std =0.02 /math .sqrt (2 *len (self .encoder )))
        for block in self .decoder :
            for weight in (block .attn .out .weight ,block .cross .out .weight ,block .ffn .down .weight ):
                nn .init .normal_ (weight ,std =0.02 /math .sqrt (3 *len (self .decoder )))

    @staticmethod 
    def _init (module ):
        if isinstance (module ,(nn .Linear ,nn .Embedding )):
            nn .init .normal_ (module .weight ,std =0.02 )

    def encode (self ,source ,source_mask ):
        x =self .embed (source )
        for block in self .encoder :
            if self .training and self .cfg ['gradient_checkpointing']:
                x =checkpoint (block ,x ,source_mask ,use_reentrant =False )
            else :
                x =block (x ,source_mask )
        return self .encoder_norm (x )

    def decode (self ,target ,memory ,source_mask ,target_mask =None ,past =None ,cross_kv =None ,use_cache =False ):
        x ,caches =self .embed (target ),[]
        for i ,block in enumerate (self .decoder ):
            if self .training and self .cfg ['gradient_checkpointing']:
                def layer (z ,mem ,module =block ):
                    return module (z ,mem ,source_mask ,target_mask )[0 ]
                x =checkpoint (layer ,x ,memory ,use_reentrant =False )
            else :
                x ,cache =block (x ,memory ,source_mask ,target_mask ,
                None if past is None else past [i ],
                None if cross_kv is None else cross_kv [i ],use_cache )
                caches .append (cache )
        return self .decoder_norm (x ),caches 

    def forward (self ,source ,target ,labels =None ):
        source_mask =source .ne (self .cfg ['pad_id'])
        memory =self .encode (source ,source_mask )
        x ,_ =self .decode (target ,memory ,source_mask ,target .ne (self .cfg ['pad_id']))
        if labels is None :
            return F .linear (x ,self .embed .weight )
        h ,y =x .flatten (0 ,1 ),labels .flatten ()
        total =h .new_zeros ((),dtype =torch .float32 )
        for start in range (0 ,len (y ),256 ):
            def part (z ,truth ):
                return F .cross_entropy (F .linear (z ,self .embed .weight ).float (),truth ,
                ignore_index =-100 ,reduction ='sum')
            if self .training :
                total =total +checkpoint (part ,h [start :start +256 ],y [start :start +256 ],use_reentrant =False )
            else :
                total =total +part (h [start :start +256 ],y [start :start +256 ])
        return total

In [10]:
class NER (LegalSeq2Seq ):
    def __init__ (self ,c ,r ):
        self .source_encoder =list (range (c ['encoder_layers']))[::2 ]if r ['student']else list (range (c ['encoder_layers']))
        self .source_decoder =list (range (c ['decoder_layers']))[::2 ]if r ['student']else list (range (c ['decoder_layers']))
        cfg =dict (c ,encoder_layers =len (self .source_encoder ),decoder_layers =len (self .source_decoder ),
        gradient_checkpointing =True )
        super ().__init__ (cfg )
        self .r =dict (r )
        self .drop =nn .Dropout (r ['dropout'])
        self .tag =nn .Linear (c ['dim'],len (TAGS ))
        self .start =nn .Linear (c ['dim'],r ['delta'])
        self .end =nn .Linear (c ['dim'],r ['delta'])

    def forward (self ,ids ,valid ):
        source_mask =valid 
        if hasattr (self ,'soft_prompt'):
            prefix =self .soft_prompt .unsqueeze (0 ).expand (ids .size (0 ),-1 ,-1 )
            memory =torch .cat ((prefix ,self .embed (ids )),dim =1 )
            source_mask =torch .cat ((valid .new_ones (ids .size (0 ),prefix .size (1 )),valid ),dim =1 )
            for block in self .encoder :
                if self .training :
                    memory =checkpoint (block ,memory ,source_mask ,use_reentrant =False )
                else :
                    memory =block (memory ,source_mask )
            memory =self .encoder_norm (memory )
        else :
            memory =self .encode (ids ,valid )
        hidden ,_ =self .decode (ids ,memory ,source_mask ,valid )
        hidden =self .drop (hidden )
        return self .tag (hidden ),self .start (hidden ),self .end (hidden )

class LoRMA (nn .Module ):
    def __init__ (self ,base ,rank ,alpha ):
        super ().__init__ ()
        self .base =base 
        self .scale =alpha /rank 
        self .a =nn .Linear (base .out_features ,rank ,bias =False )
        self .b =nn .Linear (rank ,base .out_features ,bias =False )
        nn .init .kaiming_uniform_ (self .a .weight ,a =math .sqrt (5 ))
        nn .init .zeros_ (self .b .weight )

    def forward (self ,x ):
        y =self .base (x )
        return y +self .scale *self .b (self .a (y ))

class LoRA (nn .Module ):
    def __init__ (self ,base ,rank ,alpha ):
        super ().__init__ ()
        self .base =base 
        self .scale =alpha /rank 
        self .a =nn .Linear (base .in_features ,rank ,bias =False )
        self .b =nn .Linear (rank ,base .out_features ,bias =False )
        nn .init .kaiming_uniform_ (self .a .weight ,a =math .sqrt (5 ))
        nn .init .zeros_ (self .b .weight )

    def forward (self ,x ):
        return self .base (x )+self .scale *self .b (self .a (x ))

def adapt (model ,r ):
    if r ['method']=='full':
        return 
    model .requires_grad_ (False )
    if r ['method']=='partial':
        for block in list (model .encoder )[-r ['partial_encoder']:]+list (model .decoder )[-r ['partial_decoder']:]:
            block .requires_grad_ (True )
        model .encoder_norm .requires_grad_ (True )
        model .decoder_norm .requires_grad_ (True )
    elif r ['method']in ('lorma','lora'):
        adapter =LoRMA if r ['method']=='lorma'else LoRA 
        def wrap (parent ):
            for name ,child in list (parent .named_children ()):
                if isinstance (child ,nn .Linear ):
                    setattr (parent ,name ,adapter (child ,r ['rank'],r ['alpha']))
                else :
                    wrap (child )
        wrap (model .encoder )
        wrap (model .decoder )
    elif r ['method']=='prompt':
        model .soft_prompt =nn .Parameter (torch .empty (r ['prompt_tokens'],model .cfg ['dim']))
        nn .init .normal_ (model .soft_prompt ,std =.02 )
    else :
        raise ValueError (r ['method'])
    for head in [model .tag ,model .start ,model .end ]:
        head .requires_grad_ (True )

def make_model (c ,r ,phase1 =True ):
    model =NER (c ,r )
    if phase1 :
        saved =torch .load (Path (r ['base'])/'best.pt',map_location ='cpu',weights_only =False )
        original =LegalSeq2Seq (c )
        original .load_state_dict (saved ['model'],strict =True )
        state =dict (original .state_dict ())
        if r ['student']:
            state ={k :v for k ,v in state .items ()if not k .startswith (('encoder.','decoder.'))}
            for stack ,indices in [('encoder',model .source_encoder ),('decoder',model .source_decoder )]:
                for new ,old in enumerate (indices ):
                    for key ,value in getattr (original ,stack )[old ].state_dict ().items ():
                        state [f'{stack }.{new }.{key }']=value 
        for head in ['tag','start','end']:
            for key ,value in getattr (model ,head ).state_dict ().items ():
                state [f'{head }.{key }']=value 
        model .load_state_dict (state ,strict =True )
        del saved ,original ,state 
    adapt (model ,r )
    return model

In [11]:
class Chunks (Dataset ):
    def __init__ (self ,records ,ids ,r ):
        self .records ,self .rows =records ,[]
        for k in ids :
            self .rows .extend ((k ,*w )for w in windows (len (records [k ]['ids']),r ['seq'],r ['stride']))
    def __len__ (self ):return len (self .rows )
    def __getitem__ (self ,i ):
        k ,s ,e ,a ,b =self .rows [i ];r =self .records [k ]
        target =[]
        for name in ['y','ds','de']:
            v =r [name ][s :e ].copy ();v [:a -s ]=-100 ;v [b -s :]=-100 ;target .append (v )
        return r ['ids'][s :e ],*target 

def collate (rows ):
    length =(max (len (r [0 ])for r in rows )+31 )//32 *32 
    x =torch .full ((len (rows ),length ),C ['pad_id'],dtype =torch .long )
    mask =torch .zeros_like (x ,dtype =torch .bool )
    ys =[torch .full_like (x ,-100 )for _ in range (3 )]
    for i ,row in enumerate (rows ):
        n =len (row [0 ]);x [i ,:n ]=torch .from_numpy (row [0 ]);mask [i ,:n ]=True 
        for dst ,src in zip (ys ,row [1 :]):dst [i ,:n ]=torch .from_numpy (src )
    return x ,mask ,*ys 

def loss_fn (output ,batch ,weights ,r ,teacher =None ):
    tag ,st ,en =[x .float ()for x in output ];y ,ds ,de =batch [2 :]
    main =F .cross_entropy (tag .flatten (0 ,1 ),y .flatten (),weight =weights ,ignore_index =-100 )
    boundary =tag .sum ()*0 
    for logits ,target in [(st ,ds ),(en ,de )]:
        active =target !=-100 
        boundary =boundary +F .cross_entropy (logits [active ],target [active ])if active .any ()else boundary +logits .sum ()*0 
    loss =main +r ['boundary_weight']*boundary 
    if teacher is not None :
        active =y !=-100 ;temp =r ['temperature'];soft =F .softmax (teacher [0 ][active ].float ()/temp ,-1 )
        kl =F .kl_div (F .log_softmax (tag [active ]/temp ,-1 ),soft ,reduction ='none').sum (-1 )*temp **2 
        entities =y [active ]!=0 ;parts =[kl [flag ].mean ()for flag in [entities ,~entities ]if flag .any ()]
        loss =(1 -r ['kd'])*loss +r ['kd']*torch .stack (parts ).mean ()
    return loss 

def class_weights (gold ,train_ids ,balanced ,device ):
    w =torch .ones (len (TAGS ),device =device )
    if balanced :
        count =Counter (e [2 ]for k in train_ids for e in gold [k ])
        values =np .array ([count [l ]for l in LABELS ],dtype =float )
        values =np .clip (np .sqrt (np .median (values )/np .maximum (values ,1 )),.7 ,2.5 )
        w [1 :]=torch .tensor (np .repeat (values ,2 ),device =device ,dtype =w .dtype )
    return w

In [12]:
def viterbi (logits ,bias =0. ):
    z =np .array (logits ,dtype =np .float32 ,copy =True );z [:,1 :]+=bias 
    n ,k =z .shape 
    if n ==0 :return np .empty (0 ,dtype =np .int16 )
    trans =np .zeros ((k ,k ),dtype =np .float32 )
    for j in range (2 ,k ,2 ):
        trans [:,j ]=-1e4 ;trans [j -1 :j +1 ,j ]=0 
    score =z [0 ].copy ();score [2 ::2 ]=-1e4 
    back =np .empty ((n ,k ),dtype =np .int16 )
    for i in range (1 ,n ):
        candidates =score [:,None ]+trans 
        back [i ]=candidates .argmax (0 );score =candidates [back [i ],np .arange (k )]+z [i ]
    path =np .empty (n ,dtype =np .int16 );path [-1 ]=score .argmax ()
    for i in range (n -1 ,0 ,-1 ):path [i -1 ]=back [i ,path [i ]]
    return path 

def decode (r ,outputs ,refine =True ,bias =0. ):
    tag ,st ,en =outputs ;path =viterbi (tag ,bias );spans =[];i =0 
    while i <len (path ):
        if path [i ]==0 :i +=1 ;continue 
        label =(int (path [i ])-1 )//2 ;j =i 
        while j +1 <len (path )and path [j +1 ]==2 +2 *label :j +=1 
        s ,e =int (r ['off'][i ,0 ]),int (r ['off'][j ,1 ])
        if refine :
            ns =min (st .shape [-1 ],int (r ['off'][i ,1 ]-s ))
            ne =min (en .shape [-1 ],int (e -r ['off'][j ,0 ]))
            if i ==j :
                scores =st [i ,:ns ,None ]+en [j ,None ,:ne ]
                scores =np .where (np .arange (ns )[:,None ]+np .arange (ne )[None ,:]<e -s ,scores ,-np .inf )
                a ,b =np .unravel_index (scores .argmax (),scores .shape )
            else :
                a ,b =st [i ,:ns ].argmax (),en [j ,:ne ].argmax ()
            s +=int (a );e -=int (b )
        else :
            while s <e and r ['text'][s ].isspace ():s +=1 
            while e >s and r ['text'][e -1 ].isspace ():e -=1 
        if s <e :spans .append ([s ,e ,LABELS [label ]])
        i =j +1 
    return sorted (set (map (tuple ,spans )))

def score_ner (pred ,gold ,ids ):
    counts =np .zeros ((len (LABELS ),3 ),dtype =np .int64 )
    for k in ids :
        p =set (map (tuple ,pred [k ]));g ={tuple (e [:3 ])for e in gold [k ]}
        for i ,l in enumerate (LABELS ):
            a ={x for x in p if x [2 ]==l };b ={x for x in g if x [2 ]==l }
            counts [i ]+=[len (a &b ),len (a -b ),len (b -a )]
    tp ,fp ,fn =counts .T 
    precision =tp /np .maximum (1 ,tp +fp );recall =tp /np .maximum (1 ,tp +fn )
    f1 =2 *tp /np .maximum (1 ,2 *tp +fp +fn )
    return dict (macro_f1 =float (f1 .mean ()),macro_precision =float (precision .mean ()),macro_recall =float (recall .mean ()),
    micro_f1 =float (2 *tp .sum ()/max (1 ,2 *tp .sum ()+fp .sum ()+fn .sum ())),
    per_class ={l :dict (tp =int (tp [i ]),fp =int (fp [i ]),fn =int (fn [i ]),f1 =float (f1 [i ]),precision =float (precision [i ]),recall =float (recall [i ]),support =int (tp [i ]+fn [i ]))for i ,l in enumerate (LABELS )})

@torch .inference_mode ()
def infer_doc (model ,r ,run ,device ):
    n =len (r ['ids']);dims =[len (TAGS ),run ['delta'],run ['delta']]
    sums =[np .zeros ((n ,d ),dtype =np .float32 )for d in dims ];den =np .zeros (n ,dtype =np .float32 )
    for s ,e ,_ ,_ in windows (n ,run ['seq'],run ['stride']):
        x =torch .tensor (r ['ids'][s :e ],device =device )[None ];valid =torch .ones_like (x ,dtype =torch .bool )
        with torch .autocast (device_type =device .type ,dtype =torch .float16 ,enabled =device .type =='cuda'):
            outputs =model (x ,valid )
        weight =np .minimum (np .arange (e -s )+1 ,np .arange (e -s ,0 ,-1 )).astype (np .float32 )
        weight =np .minimum (weight ,128 )/128 
        for total ,z in zip (sums ,outputs ):total [s :e ]+=z [0 ].float ().cpu ().numpy ()*weight [:,None ]
        den [s :e ]+=weight 
    return [z /den [:,None ]for z in sums ]

@torch .inference_mode ()
def validate (model ,records ,gold ,ids ,run ,device ):
    model .eval ();pred ={k :decode (records [k ],infer_doc (model ,records [k ],run ,device ))for k in ids }
    return score_ner (pred ,gold ,ids ),pred

## 4. Two-GPU training

In [13]:
def launch (fn ,*args ):
    payload =OUT /'worker.pkl'
    with payload .open ('wb')as f :cloudpickle .dump ((fn ,args ),f )
    with socket .socket ()as s :
        s .bind (('127.0.0.1',0 ));port =s .getsockname ()[1 ]
    command ="import cloudpickle,os; f=open(os.environ['WORKER_PAYLOAD'],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(int(os.environ['RANK']),*args)"
    jobs =[]
    for rank in range (2 ):
        env =dict (os .environ ,MASTER_ADDR ='127.0.0.1',MASTER_PORT =str (port ),RANK =str (rank ),
        WORKER_PAYLOAD =str (payload ),OMP_NUM_THREADS ='2',TOKENIZERS_PARALLELISM ='false')
        jobs .append (subprocess .Popen ([sys .executable ,'-u','-c',command ],env =env ))
    while any (p .poll ()is None for p in jobs ):
        if any (p .poll ()not in (None ,0 )for p in jobs ):
            for p in jobs :
                if p .poll ()is None :p .terminate ()
            raise RuntimeError ('GPU worker failed; see its traceback above.')
        time .sleep (.5 )
    if any (p .returncode for p in jobs ):raise RuntimeError ('GPU worker failed.')

In [14]:
def train_worker (rank ,run ,resume =False ):
    torch .set_num_threads (2 );torch .cuda .set_device (rank )
    dist .init_process_group ('nccl',rank =rank ,world_size =2 ,device_id =torch .device ('cuda',rank ),timeout =timedelta (minutes =60 ))
    device =torch .device ('cuda',rank );out =Path (run ['out'])/run ['name'];out .mkdir (exist_ok =True )
    with open (run ['prepared'],'rb')as f :pack =cloudpickle .load (f )
    records ,gold ,c =pack ['records'],pack ['gold'],pack ['config']
    tr ,va =pack ['split']['train'],pack ['split']['valid']
    if run .get ('refit'):tr =sorted (gold );va =[]
    torch .manual_seed (run ['seed']);np .random .seed (run ['seed']+rank );random .seed (run ['seed']+rank )
    model =make_model (c ,run ,phase1 =not resume ).to (device )
    teacher =None 
    if run ['teacher']:
        tc =torch .load (run ['teacher'],map_location ='cpu',weights_only =False )
        if tc ['manifest']!=pack ['manifest']:raise ValueError ('Teacher data/split differs from the student.')
        if tc ['run'].get ('refit')and not run .get ('refit'):
            raise ValueError ('A refitted teacher has seen validation; use a training-only teacher.')
        teacher =make_model (tc ['config'],tc ['run'],phase1 =False )
        teacher .load_state_dict (tc ['model']);teacher .to (device ).eval ().requires_grad_ (False );del tc 
    groups =[]
    for head in [False ,True ]:
        for decay in [False ,True ]:
            ps =[p for name ,p in model .named_parameters ()if p .requires_grad and 
            name .startswith (('tag.','start.','end.'))==head and (p .ndim >=2 )==decay ]
            if ps :groups .append (dict (params =ps ,lr =run ['head_lr']if head else run ['lr'],
            initial_lr =run ['head_lr']if head else run ['lr'],weight_decay =run ['wd']if decay else 0. ))
    opt =torch .optim .AdamW (groups ,betas =(.9 ,.999 ),eps =1e-8 ,fused =True )
    scaler =torch .amp .GradScaler ('cuda',init_scale =1024 )
    dataset =Chunks (records ,tr ,run )
    sampler =DistributedSampler (dataset ,num_replicas =2 ,rank =rank ,shuffle =True ,seed =run ['seed'])
    loader =DataLoader (dataset ,batch_size =run ['micro'],sampler =sampler ,collate_fn =collate ,
    num_workers =0 ,pin_memory =True )
    weights =class_weights (gold ,tr ,run ['balanced'],device )
    start_epoch ,step ,best ,wait ,best_epoch =0 ,0 ,-1. ,0 ,0 
    torch .manual_seed (run ['seed']+rank )
    if resume :
        ck =torch .load (out /'last.pt',map_location ='cpu',weights_only =False )
        if ck ['manifest']!=pack ['manifest']:raise ValueError ('Resume data/tokenizer/split changed.')
        for key in ['seq','stride','epochs','causal','method','student','balanced','seed','teacher']:
            if ck ['run'][key ]!=run [key ]:raise ValueError (f'Resume configuration changed: {key }')
        model .load_state_dict (ck ['model']);opt .load_state_dict (ck ['optimizer']);scaler .load_state_dict (ck ['scaler'])
        start_epoch ,step ,best ,wait ,best_epoch =ck ['epoch'],ck ['step'],ck ['best'],ck ['wait'],ck ['best_epoch']
        torch .set_rng_state (ck ['rng'][rank ]['cpu']);torch .cuda .set_rng_state (ck ['rng'][rank ]['cuda'],device )
        np .random .set_state (ck ['rng'][rank ]['numpy']);random .setstate (ck ['rng'][rank ]['python']);del ck 
    ddp =DDP (model ,device_ids =[rank ],broadcast_buffers =False ,gradient_as_bucket_view =True )
    total =math .ceil (len (loader )/run ['accum'])*run ['epochs'];warm =max (1 ,int (total *run ['warmup']))
    if rank ==0 :
        print (f"{run ['name']}: {sum (p .numel ()for p in model .parameters ()):,} parameters | "
        f"{sum (p .numel ()for p in model .parameters ()if p .requires_grad ):,} trainable | {len (dataset )} windows",flush =True )
        (out /'run.json').write_text (json .dumps (run ,indent =2 ))
    for epoch in range (start_epoch ,run ['stop_epoch']):
        if va and wait >=run ['patience']:break 
        sampler .set_epoch (epoch );ddp .train ();opt .zero_grad (set_to_none =True )
        torch .cuda .reset_peak_memory_stats ();started =time .monotonic ();running =torch .zeros ((),device =device )
        for i ,cpu_batch in enumerate (loader ):
            batch =[x .to (device ,non_blocking =True )for x in cpu_batch ]
            size =min (run ['accum'],len (loader )-(i //run ['accum'])*run ['accum'])
            sync =(i +1 )%run ['accum']==0 or i +1 ==len (loader )
            ratio =(step +1 )/warm if step <warm else .1 +.9 *.5 *(1 +math .cos (math .pi *min (1. ,(step -warm )/max (1 ,total -warm ))))
            for group in opt .param_groups :group ['lr']=group ['initial_lr']*ratio 
            with nullcontext ()if sync else ddp .no_sync ():
                with torch .autocast ('cuda',dtype =torch .float16 ):
                    soft =None 
                    if teacher is not None :
                        with torch .no_grad ():soft =teacher (*batch [:2 ])
                    outputs =ddp (*batch [:2 ]);loss =loss_fn (outputs ,batch ,weights ,run ,soft )
                scaler .scale (loss /size ).backward ()
            running +=loss .detach ()
            if sync :
                scaler .unscale_ (opt );torch .nn .utils .clip_grad_norm_ (model .parameters (),1. )
                old_scale =scaler .get_scale ();scaler .step (opt );scaler .update ()
                if scaler .get_scale ()>=old_scale :step +=1 
                opt .zero_grad (set_to_none =True )
        model .eval ();torch .cuda .synchronize ();seconds =time .monotonic ()-started 
        peak =torch .tensor (torch .cuda .max_memory_allocated ()/2 **30 ,device =device )
        dist .all_reduce (peak ,op =dist .ReduceOp .MAX );dist .all_reduce (running )
        local ={k :decode (records [k ],infer_doc (model ,records [k ],run ,device ))for k in va [rank ::2 ]}
        gathered =[None ,None ];dist .all_gather_object (gathered ,local )
        pred ={k :v for part in gathered for k ,v in part .items ()}
        metrics =score_ner (pred ,gold ,va )if va else dict (macro_f1 =0. )
        improved =metrics ['macro_f1']>best or not va 
        if improved :best =metrics ['macro_f1'];best_epoch =epoch +1 ;wait =0 
        else :wait +=1 
        state =dict (cpu =torch .get_rng_state (),cuda =torch .cuda .get_rng_state (device ),
        numpy =np .random .get_state (),python =random .getstate ())
        states =[None ,None ];dist .all_gather_object (states ,state )
        if rank ==0 :
            log =dict (epoch =epoch +1 ,step =step ,loss =float (running )/(2 *len (loader )),
            seconds =seconds ,peak_gib =float (peak ),metrics =metrics ,
            total_parameters =sum (p .numel ()for p in model .parameters ()),
            trainable_parameters =sum (p .numel ()for p in model .parameters ()if p .requires_grad ))
            with (out /'history.jsonl').open ('a'if resume or epoch else 'w')as f :f .write (json .dumps (log )+'\n')
            print (f"{run ['name']} epoch {epoch +1 } | loss {log ['loss']:.4f} | macro-F1 {metrics ['macro_f1']:.4f} | {seconds /60 :.1f} min | {float (peak ):.2f} GiB",flush =True )
            ck =dict (model =model .state_dict (),config =c ,run =run ,labels =pack ['labels'],epoch =epoch +1 ,step =step ,best =best ,
            best_epoch =best_epoch ,wait =wait ,metrics =metrics ,manifest =pack ['manifest'])
            if improved :
                best_ck =dict (ck ,model ={k :v .detach ().cpu ().half ()if v .is_floating_point ()else v .detach ().cpu ()for k ,v in model .state_dict ().items ()})
                torch .save (best_ck ,out /'best.tmp');os .replace (out /'best.tmp',out /'best.pt')
                del best_ck 
                (out /'valid_predictions.json').write_text (json .dumps (pred ))
                (out /'best_metrics.json').write_text (json .dumps (dict (epoch =best_epoch ,**metrics ),indent =2 ))
            ck .update (optimizer =opt .state_dict (),scaler =scaler .state_dict (),rng =states )
            torch .save (ck ,out /'last.tmp');os .replace (out /'last.tmp',out /'last.pt')
        dist .barrier ()
    if rank ==0 :
        (out /'completed.json').write_text (json .dumps (dict (best_epoch =best_epoch ,best =best )))
        (out /'last.pt').unlink (missing_ok =True )
    dist .barrier ()
    dist .destroy_process_group ()

## 5. Adaptation methods

In [15]:
METHODS =['lorma','full','partial','lora','prompt']

def run_experiment (run ):
    folder =OUT /run ['name']
    if (folder /'completed.json').exists ():
        return 
    launch (train_worker ,run ,False )

for method in METHODS :
    run =dict (R ,name =method ,method =method ,stop_epoch =R ['epochs'],
    lr ={'lorma':2e-4 ,'full':2e-5 ,'partial':3e-5 ,'lora':2e-4 ,'prompt':1e-3 }[method ])
    run_experiment (run )

CANDIDATES =[str (OUT /m /'best.pt')for m in METHODS 
if (OUT /m /'completed.json').exists ()]

lorma: 178,428,729 parameters | 2,820,921 trainable | 1201 windows
lorma epoch 1 | loss 1.0658 | macro-F1 0.1893 | 2.3 min | 0.82 GiB
lorma epoch 2 | loss 0.1503 | macro-F1 0.3290 | 2.4 min | 0.82 GiB
lorma epoch 3 | loss 0.1028 | macro-F1 0.3811 | 2.4 min | 0.82 GiB
lorma epoch 4 | loss 0.0792 | macro-F1 0.3784 | 2.4 min | 0.82 GiB
lorma epoch 5 | loss 0.0638 | macro-F1 0.4124 | 2.4 min | 0.82 GiB
lorma epoch 6 | loss 0.0494 | macro-F1 0.3895 | 2.4 min | 0.82 GiB
lorma epoch 7 | loss 0.0357 | macro-F1 0.4785 | 2.4 min | 0.82 GiB
lorma epoch 8 | loss 0.0263 | macro-F1 0.4787 | 2.4 min | 0.82 GiB
lorma epoch 9 | loss 0.0178 | macro-F1 0.4563 | 2.4 min | 0.82 GiB
lorma epoch 10 | loss 0.0130 | macro-F1 0.4343 | 2.4 min | 0.82 GiB
lorma epoch 11 | loss 0.0101 | macro-F1 0.4441 | 2.4 min | 0.82 GiB
full: 175,651,641 parameters | 175,651,641 trainable | 1201 windows
full epoch 1 | loss 0.9653 | macro-F1 0.2899 | 1.8 min | 3.66 GiB
full epoch 2 | loss 0.1210 | macro-F1 0.3836 | 1.9 min | 3.6

## 6. Knowledge distillation

In [16]:
teacher =max (CANDIDATES ,key =lambda p :json .loads ((Path (p ).parent /'best_metrics.json').read_text ())['macro_f1'])
for use_kd in [False ,True ]:
    name ='student_kd'if use_kd else 'student_gold'
    run =dict (R ,name =name ,method ='full',student =True ,lr =3e-5 ,
    teacher =teacher if use_kd else None ,stop_epoch =R ['epochs'])
    run_experiment (run )
    if use_kd :
        CANDIDATES .append (str (OUT /name /'best.pt'))

student_gold: 100,136,505 parameters | 100,136,505 trainable | 1201 windows
student_gold epoch 1 | loss 1.0510 | macro-F1 0.2467 | 1.0 min | 2.08 GiB
student_gold epoch 2 | loss 0.1299 | macro-F1 0.3647 | 1.0 min | 2.08 GiB
student_gold epoch 3 | loss 0.0627 | macro-F1 0.4034 | 1.0 min | 2.08 GiB
student_gold epoch 4 | loss 0.0303 | macro-F1 0.4027 | 1.0 min | 2.08 GiB
student_gold epoch 5 | loss 0.0151 | macro-F1 0.4710 | 1.0 min | 2.08 GiB
student_gold epoch 6 | loss 0.0082 | macro-F1 0.4855 | 1.0 min | 2.08 GiB
student_gold epoch 7 | loss 0.0036 | macro-F1 0.4351 | 1.0 min | 2.08 GiB
student_gold epoch 8 | loss 0.0017 | macro-F1 0.4574 | 1.0 min | 2.08 GiB
student_gold epoch 9 | loss 0.0008 | macro-F1 0.4719 | 1.0 min | 2.08 GiB
student_kd: 100,136,505 parameters | 100,136,505 trainable | 1201 windows
student_kd epoch 1 | loss 2.7337 | macro-F1 0.2493 | 1.3 min | 2.74 GiB
student_kd epoch 2 | loss 0.4109 | macro-F1 0.4020 | 1.3 min | 2.74 GiB
student_kd epoch 3 | loss 0.2029 | macro

## 7. Decode selection

In [17]:
def cache_worker (rank ,path ,ids ,destination ):
    torch .set_num_threads (2 );torch .cuda .set_device (rank );device =torch .device ('cuda',rank )
    ck =torch .load (path ,map_location ='cpu',weights_only =False );run =ck ['run']
    model =make_model (ck ['config'],run ,phase1 =False );model .load_state_dict (ck ['model']);model .to (device ).eval ()
    with open (R ['prepared'],'rb')as f :pack =cloudpickle .load (f )
    if ck ['manifest']!=pack ['manifest']:raise ValueError ('Inference inputs differ from the checkpoint.')
    records =pack ['records']
    arrays ={};torch .cuda .reset_peak_memory_stats ();torch .cuda .synchronize ();start =time .monotonic ()
    for k in ids [rank ::2 ]:
        for j ,z in enumerate (infer_doc (model ,records [k ],run ,device )):arrays [k +'__'+str (j )]=z .astype (np .float16 )
    torch .cuda .synchronize ()
    np .savez_compressed (f'{destination }.{rank }.npz',**arrays )
    Path (f'{destination }.{rank }.json').write_text (json .dumps (dict (seconds =time .monotonic ()-start ,
    peak_gib =torch .cuda .max_memory_allocated ()/2 **30 ,documents =len (ids [rank ::2 ]))))

comparison =[];selected_score =-1. ;selected =None 
selections ={}
for path in CANDIDATES :
    method =Path (path ).parent .name 
    method_score =-1. 
    dest =Path (path ).parent /'valid_logits'
    launch (cache_worker ,path ,valid_ids ,str (dest ))
    cached ={}
    for rank in range (2 ):
        with np .load (f'{dest }.{rank }.npz')as z :cached .update ({k :z [k ].astype (np .float32 )for k in z .files })
    for refine in [False ,True ]:
        for bias in [-.4 ,0. ,.4 ]:
            pred ={k :decode (records [k ],[cached [k +'__'+str (j )]for j in range (3 )],refine ,bias )for k in valid_ids }
            metrics =score_ner (pred ,gold ,valid_ids )
            entry =dict (checkpoint =path ,refine =refine ,bias =bias ,**metrics );comparison .append (entry )
            if metrics ['macro_f1']>method_score :
                method_score =metrics ['macro_f1']
                selections [method ]=entry 
                (Path (path ).parent /'selected_valid_predictions.json').write_text (json .dumps (pred ))
            if metrics ['macro_f1']>selected_score :
                selected_score =metrics ['macro_f1'];selected =entry ;selected_pred =pred 
    for rank in range (2 ):
        Path (f'{dest }.{rank }.npz').unlink (missing_ok =True )
(OUT /'comparison.json').write_text (json .dumps (comparison ,indent =2 ))
(OUT /'selection.json').write_text (json .dumps (selected ,indent =2 ))
(OUT /'selected_valid_predictions.json').write_text (json .dumps (selected_pred ))
print (f"Selected: {Path (selected ['checkpoint']).parent .name } | refined={selected ['refine']} | bias={selected ['bias']} | strict macro-F1={selected_score :.4f}")
for label ,row in selected ['per_class'].items ():print (label ,round (row ['f1'],4 ),'support',row ['support'])
(OUT /'method_selections.json').write_text (json .dumps (selections ,indent =2 ))

Selected: full | refined=True | bias=0.4 | strict macro-F1=0.5160
APP 0.2941 support 81
RESP 0.3333 support 73
A.COUNSEL 0.3636 support 30
R.COUNSEL 0.6353 support 40
JUDGE 0.5672 support 57
WIT 0.4286 support 35
AUTH 0.35 support 198
COURT 0.8257 support 223
STAT 0.6137 support 228
PREC 0.4542 support 241
DATE 0.9327 support 305
CASENO 0.3935 support 187


17210

In [18]:
errors =Counter ();examples =[]
for k in valid_ids :
    p =set (map (tuple ,selected_pred [k ]));g ={tuple (e [:3 ])for e in gold [k ]}
    for span in p -g :
        s ,e ,l =span 
        kind =('wrong_label'if any ((s ,e )==x [:2 ]for x in g )else 
        'boundary'if any (l ==x [2 ]and s <x [1 ]and e >x [0 ]for x in g )else 'false_positive')
        errors [kind ]+=1 
        if len (examples )<40 :examples .append (dict (id =k ,type =kind ,span =span ,text =texts [k ][s :e ]))
    errors ['missed_gold']+=len (g -p )
(OUT /'errors.json').write_text (json .dumps (dict (counts =dict (errors ),examples =examples ),indent =2 ))
print ('Validation errors:',dict (errors ))

rng =np .random .default_rng (1781 );boot =[]
for _ in range (500 ):
    sampled =rng .choice (valid_ids ,len (valid_ids ),replace =True ).tolist ()
    boot .append (score_ner (selected_pred ,gold ,sampled )['macro_f1'])
interval =np .quantile (boot ,[.025 ,.975 ]).tolist ()
(OUT /'validation_uncertainty.json').write_text (json .dumps (dict (document_bootstrap_95 =interval ,
note ='Conditional on model selection on the same 14 documents; not an unbiased generalization interval.')))
print ('Document-bootstrap interval:',[round (x ,4 )for x in interval ])

Validation errors: {'false_positive': 362, 'boundary': 289, 'missed_gold': 694, 'wrong_label': 70}
Document-bootstrap interval: [0.4468, 0.5714]


## 8. Submissions

In [21]:
submission_paths ={method :entry ['checkpoint']for method ,entry in selections .items ()}
for method in METHODS +['student_kd']:
    ck =torch .load (selections [method ]['checkpoint'],map_location ='cpu',weights_only =False )
    run =dict (ck ['run'],name =method +'_refit',refit =True ,stop_epoch =ck ['epoch'])
    if run ['teacher']:
        teacher_method =Path (run ['teacher']).parent .name 
        run ['teacher']=submission_paths [teacher_method ]
    del ck 
    run_experiment (run )
    submission_paths [method ]=str (OUT /run ['name']/'best.pt')

submission_dir =OUT /'submissions'
submission_dir .mkdir (exist_ok =True )
submission_manifest ={}
for method in METHODS +['student_kd']:
    settings =selections [method ]
    path =submission_paths [method ]
    dest =OUT /method /'test_logits'
    launch (cache_worker ,path ,list (test ),str (dest ))
    cached ={}
    for rank in range (2 ):
        with np .load (f'{dest }.{rank }.npz')as z :
            cached .update ({k :z [k ].astype (np .float32 )for k in z .files })
    prediction ={k :decode (records [k ],[cached [k +'__'+str (j )]for j in range (3 )],
    settings ['refine'],settings ['bias'])for k in test }
    json_path =submission_dir /f'{method }.json'
    json_path .write_text (json .dumps (prediction ,ensure_ascii =False ))
    zip_path =submission_dir /f'{method }.zip'
    with zipfile .ZipFile (zip_path ,'w',zipfile .ZIP_DEFLATED )as archive :
        archive .write (json_path ,'prediction.json')
    submission_manifest [method ]=dict (checkpoint =path ,zip =str (zip_path ),
    refine =settings ['refine'],bias =settings ['bias'],refit =True ,
    validation_macro_f1 =settings ['macro_f1'],documents =len (prediction ))
    print (f"{method }: {len (prediction )} documents | {zip_path }")
    for rank in range (2 ):
        Path (f'{dest }.{rank }.npz').unlink (missing_ok =True )
    del cached 
(OUT /'submission_manifest.json').write_text (json .dumps (submission_manifest ,indent =2 ))

lorma_refit: 178,428,729 parameters | 2,820,921 trainable | 1445 windows
lorma_refit epoch 1 | loss 0.9756 | macro-F1 0.0000 | 2.9 min | 0.82 GiB
lorma_refit epoch 2 | loss 0.1472 | macro-F1 0.0000 | 2.9 min | 0.82 GiB
lorma_refit epoch 3 | loss 0.1035 | macro-F1 0.0000 | 2.9 min | 0.82 GiB
lorma_refit epoch 4 | loss 0.0850 | macro-F1 0.0000 | 2.9 min | 0.82 GiB
lorma_refit epoch 5 | loss 0.0648 | macro-F1 0.0000 | 2.9 min | 0.82 GiB
lorma_refit epoch 6 | loss 0.0498 | macro-F1 0.0000 | 2.9 min | 0.82 GiB
lorma_refit epoch 7 | loss 0.0370 | macro-F1 0.0000 | 3.0 min | 0.82 GiB
lorma_refit epoch 8 | loss 0.0266 | macro-F1 0.0000 | 3.0 min | 0.82 GiB
full_refit: 175,651,641 parameters | 175,651,641 trainable | 1445 windows
full_refit epoch 1 | loss 0.8872 | macro-F1 0.0000 | 2.3 min | 3.66 GiB
full_refit epoch 2 | loss 0.1185 | macro-F1 0.0000 | 2.3 min | 3.64 GiB
full_refit epoch 3 | loss 0.0622 | macro-F1 0.0000 | 2.3 min | 3.64 GiB
full_refit epoch 4 | loss 0.0324 | macro-F1 0.0000 | 

1943

## 9. Checkpoints and results

In [22]:
result =dict (backbone ='P1_S3_EncoderDecoder176M',
checkpoint_format ='complete_ner_state_fp16',
submissions =submission_manifest ,selections =selections ,
base_phase1_sha256 =manifest ['base_checkpoint_sha256'])
(OUT /'phase2_result.json').write_text (json .dumps (result ,indent =2 ))
print ('Six submission ZIPs:',submission_dir )

Six submission ZIPs: /kaggle/working/phase2_encoder_decoder_six_methods/submissions


In [23]:
from collections import Counter 
import json 
import zipfile 

ensemble_methods =['full','partial','lora','lorma','prompt','student_kd']

method_scores =json .loads ((OUT /'method_selections.json').read_text ())
valid_predictions ={
m :json .loads ((OUT /m /'selected_valid_predictions.json').read_text ())
for m in ensemble_methods 
}
test_predictions ={
m :json .loads ((OUT /'submissions'/f'{m }.json').read_text ())
for m in ensemble_methods 
}

def vote_predictions (predictions ,methods ,document_ids ,min_votes ):
    result ={}
    for doc_id in document_ids :
        votes =Counter ()
        supporters ={}

        for method in methods :
            spans ={tuple (row [:3 ])for row in predictions [method ][doc_id ]}
            votes .update (spans )
            for span in spans :
                supporters [span ]=max (
                supporters .get (span ,0. ),
                method_scores [method ]['macro_f1']
                )

        candidates =sorted (
        (span for span ,count in votes .items ()if count >=min_votes ),
        key =lambda span :(-votes [span ],-supporters [span ],span )
        )

        kept =[]
        for start ,end ,label in candidates :
            if not any (start <b and end >a for a ,b ,_ in kept ):
                kept .append ((start ,end ,label ))

        result [doc_id ]=[list (span )for span in sorted (kept )]

    return result

In [24]:
ensemble_trials =[]

for min_votes in [2 ,3 ,4 ]:
    pred =vote_predictions (
    valid_predictions ,ensemble_methods ,valid_ids ,min_votes 
    )
    metrics =score_ner (pred ,gold ,valid_ids )
    ensemble_trials .append (dict (min_votes =min_votes ,**metrics ))

ensemble_choice =max (ensemble_trials ,key =lambda row :row ['macro_f1'])

(OUT /'ensemble_selection.json').write_text (json .dumps (
dict (methods =ensemble_methods ,selected =ensemble_choice ,
trials =ensemble_trials ),
indent =2 
))

best_single =max (method_scores [m ]['macro_f1']for m in ensemble_methods )
print (f"Best individual: {best_single :.4f}")
print (f"Ensemble: {ensemble_choice ['macro_f1']:.4f} "
f"| minimum votes: {ensemble_choice ['min_votes']}")

Best individual: 0.5160
Ensemble: 0.5512 | minimum votes: 2


In [25]:
ensemble_prediction =vote_predictions (
test_predictions ,ensemble_methods ,list (test ),
ensemble_choice ['min_votes']
)

ensemble_dir =OUT /'submissions'
json_path =ensemble_dir /'ensemble.json'
zip_path =ensemble_dir /'ensemble.zip'

json_path .write_text (json .dumps (ensemble_prediction ,ensure_ascii =False ))

with zipfile .ZipFile (zip_path ,'w',zipfile .ZIP_DEFLATED )as archive :
    archive .write (json_path ,'prediction.json')

print ('Ensemble submission:',zip_path )

Ensemble submission: /kaggle/working/phase2_encoder_decoder_six_methods/submissions/ensemble.zip


## Interpretation